**Final Project:** Data Transformation <br>
**Date:** 04/14/2026 <br>
**Group 4:** Erin Wright-Vazquez | Jacob Ko | Sadia Rahman <br>

#### Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import time
from pybaseball import pitching_stats
from pybaseball import statcast
import openmeteo_requests
import requests_cache
from retry_requests import retry
import requests
from pybaseball import playerid_reverse_lookup
import psycopg2
from io import StringIO

In [2]:
def clean_name(name):
    
    name = name.lower() # convert to lowercase
    name = re.sub(r"[^\w\s]", "", name) # remove punctuation/special characters
    name = re.sub(r"\s+", " ", name) # normalize whitespace
    name = name.strip() # remove leading/trailing spaces
    
    return name

#### Seamshead

In [3]:
# load the dataset
seamheads = pd.read_csv("./data-sources/Parks.csv") 

# basic structure
print(seamheads.shape)
print(seamheads.columns)
print(seamheads.head()) # preview rows
print(seamheads.dtypes) # data types
print(seamheads.isnull().sum()) # missing values

(262, 13)
Index(['PARKID', 'NAME', 'CITY', 'STATE', 'START', 'END', 'LEAGUE', 'NOTES',
       'AKA', 'Exact', 'Latitude', 'Longitude', 'Altitude'],
      dtype='object')
  PARKID                      NAME         CITY STATE       START         END  \
0  ALB01            Riverside Park  East Albany    NY  18800911.0  18820530.0   
1  ALT01             Columbia Park      Altoona    PA  18840430.0  18840531.0   
2  ANA01  Angel Stadium of Anaheim      Anaheim    CA  19660419.0         NaN   
3  ARL01         Arlington Stadium    Arlington    TX  19720421.0  19931003.0   
4  ARL02           Globe Life Park    Arlington    TX  19940411.0  20190929.0   

  LEAGUE                                           NOTES  \
0     NL  TRN:9/11/80;6/15&9/10/1881;5/16-5/18&5/30/1882   
1     UA                                             NaN   
2     AL                                             NaN   
3     AL                                             NaN   
4     AL                                   

In [4]:
# renaming columns to snake_case for standardization and only keeping columns we'd need
seamheads = (seamheads.rename(columns = {"PARKID": "park_id",
                                         "NAME": "park_name",
                                         "CITY": "city",
                                         "STATE": "state",
                                         "Latitude": "latitude",
                                         "Longitude": "longitude"
                                         })
             [["park_id", "park_name", "city", "state", "latitude", "longitude" ]]
             )

seamheads.head()

,park_id,park_name,city,state,latitude,longitude
0,ALB01,Riverside Park,East Albany,NY,42.642295,-73.745198
1,ALT01,Columbia Park,Altoona,PA,40.494582,-78.411164
2,ANA01,Angel Stadium of Anaheim,Anaheim,CA,33.800290,-117.882685
3,ARL01,Arlington Stadium,Arlington,TX,32.755974,-97.084778
4,ARL02,Globe Life Park,Arlington,TX,32.751164,-97.082546


In [5]:
seamheads["park_name_clean"] = seamheads["park_name"].apply(clean_name) # clean park names for matching across datasets
seamheads["park_grouping"] = seamheads["park_name_clean"].str.replace(" ", "_") # replace spaces with underscores to create a consistent key
seamheads.head()

,park_id,park_name,city,state,latitude,longitude,park_name_clean,park_grouping
0,ALB01,Riverside Park,East Albany,NY,42.642295,-73.745198,riverside park,riverside_park
1,ALT01,Columbia Park,Altoona,PA,40.494582,-78.411164,columbia park,columbia_park
2,ANA01,Angel Stadium of Anaheim,Anaheim,CA,33.800290,-117.882685,angel stadium of anaheim,angel_stadium_of_anaheim
3,ARL01,Arlington Stadium,Arlington,TX,32.755974,-97.084778,arlington stadium,arlington_stadium
4,ARL02,Globe Life Park,Arlington,TX,32.751164,-97.082546,globe life park,globe_life_park


In [6]:
# check uniqueness and duplicates
print("rows:", len(seamheads))
print("unique park_id:", seamheads["park_id"].nunique())
print("unique park_key:", seamheads["park_grouping"].nunique())

# show any duplicated park_id
dupe_id = seamheads[seamheads.duplicated("park_id", keep = False)].sort_values("park_id")
print("duplicate park_id rows:", len(dupe_id))
display(dupe_id[["park_id","park_name","city","state"]].head(50))

# show any duplicated park_key 
dupe_key = seamheads[seamheads.duplicated("park_grouping", keep = False)].sort_values("park_grouping")
print("duplicate park_key rows:", len(dupe_key))
display(dupe_key[["park_id","park_name","city","state"]].head(80))

rows: 262
unique park_id: 262
unique park_key: 256
duplicate park_id rows: 0


,park_id,park_name,city,state


duplicate park_key rows: 10


,park_id,park_name,city,state
107,KAN01,Athletic Park,Kansas City,MO
136,MIN01,Athletic Park,Minneapolis,MN
244,WAS04,Athletic Park,Washington,DC
1,ALT01,Columbia Park,Altoona,PA
180,PHI10,Columbia Park,Philadelphia,PA
74,DET01,Recreation Park,Detroit,MI
174,PHI04,Recreation Park,Philadelphia,PA
189,PIT04,Recreation Park,Pittsburgh,PA
45,CHI11,Wrigley Field,Chicago,IL
118,LOS02,Wrigley Field,Los Angeles,CA


In [7]:
# final cleaned seamheads dataset
ballpark_1nf = seamheads[["park_id",
                          "park_name",
                          "park_name_clean",
                          "city",
                          "state",
                          "latitude",
                          "longitude",
                          "park_grouping"
                          ]].copy()

ballpark_1nf.head()

,park_id,park_name,park_name_clean,city,state,latitude,longitude,park_grouping
0,ALB01,Riverside Park,riverside park,East Albany,NY,42.642295,-73.745198,riverside_park
1,ALT01,Columbia Park,columbia park,Altoona,PA,40.494582,-78.411164,columbia_park
2,ANA01,Angel Stadium of Anaheim,angel stadium of anaheim,Anaheim,CA,33.800290,-117.882685,angel_stadium_of_anaheim
3,ARL01,Arlington Stadium,arlington stadium,Arlington,TX,32.755974,-97.084778,arlington_stadium
4,ARL02,Globe Life Park,globe life park,Arlington,TX,32.751164,-97.082546,globe_life_park


The Seamheads dataset serves as the foundational source for ballpark information. Initial preprocessing focuses on standardizing column names into snake_case and filtering only relevant fields such as park ID, name, location, and coordinates. A cleaning function is applied to normalize park names by removing punctuation and enforcing lowercase formatting. This enables consistent matching across datasets. To support integration, a derived field called park_grouping is created by replacing spaces with underscores in cleaned park names. This acts as a standardized join key across datasets. Data quality checks confirm that park_id is unique across all records, while minor duplication exists in the grouping key due to historically reused park names in different cities.

#### MLB (Kaggle)

In [8]:
kaggle = pd.read_csv("./data-sources/ballparks.csv")

# preview quality and structure of data
print(kaggle.columns.tolist())
print(kaggle.shape)
print(kaggle.head())
print(kaggle.dtypes)
print(kaggle.isnull().sum())
print(kaggle.duplicated().sum())
print(kaggle.duplicated(subset = ["team_name"]).sum())
print(kaggle.duplicated(subset = ["ballpark"]).sum())
print(kaggle[kaggle.duplicated(subset = ["ballpark"], keep = False)])
kaggle.describe()

['team_name', 'ballpark', 'left_field', 'center_field', 'right_field', 'min_wall_height', 'max_wall_height', 'hr_park_effects', 'extra_distance', 'avg_temp', 'elevation', 'roof', 'daytime']
(30, 13)
  team_name                     ballpark  left_field  center_field  \
0       ATL                  Truist Park         335           400   
1        AZ                  Chase Field         328           407   
2       BAL  Oriole Park at Camden Yards         333           400   
3       BOS                  Fenway Park         310           420   
4       CHC                Wrigley Field         355           400   

   right_field  min_wall_height  max_wall_height  hr_park_effects  \
0          325             11.0               15               99   
1          335              7.6               25               84   
2          318              7.0               21              107   
3          302              3.0               37              102   
4          353             11.5    

,left_field,center_field,right_field,min_wall_height,max_wall_height,hr_park_effects,extra_distance,avg_temp,elevation,roof,daytime
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,331.833333,404.166667,328.333333,7.553333,14.266667,100.500000,-0.030000,73.480000,517.466667,0.185333,0.383333
std,9.857898,6.264753,10.459555,1.797265,7.060982,16.812865,5.966004,3.943602,948.785384,0.350917,0.086715
min,310.000000,395.000000,302.000000,3.000000,8.000000,75.000000,-7.700000,63.800000,0.000000,0.000000,0.240000
25%,328.000000,400.000000,325.000000,7.000000,8.000000,88.250000,-3.700000,71.300000,21.500000,0.000000,0.320000
50%,330.000000,404.000000,329.000000,8.000000,12.000000,99.000000,-1.250000,73.350000,365.000000,0.000000,0.380000
75%,336.000000,407.000000,333.750000,8.000000,18.500000,111.750000,1.250000,76.475000,599.250000,0.120000,0.417500
max,355.000000,420.000000,353.000000,11.500000,37.000000,150.000000,22.000000,80.800000,5190.000000,1.000000,0.610000


In [9]:
# clean and standardize Kaggle dataset keys
kaggle["park_name_clean"] = kaggle["ballpark"].apply(clean_name)
kaggle["park_grouping"] = kaggle["park_name_clean"].str.replace(" ", "_", regex = False)

# clean team names and create join key
kaggle["team_name_clean"] = kaggle["team_name"].apply(clean_name)
kaggle["team_key"] = kaggle["team_name_clean"].str.replace(" ", "_", regex = False)

In [10]:
kaggle_clean = kaggle[["team_name",
                       "team_key",
                       "ballpark",
                       "park_name_clean",
                       "park_grouping",
                       ]].copy()

kaggle_clean.head()

,team_name,team_key,ballpark,park_name_clean,park_grouping
0,ATL,atl,Truist Park,truist park,truist_park
1,AZ,az,Chase Field,chase field,chase_field
2,BAL,bal,Oriole Park at Camden Yards,oriole park at camden yards,oriole_park_at_camden_yards
3,BOS,bos,Fenway Park,fenway park,fenway_park
4,CHC,chc,Wrigley Field,wrigley field,wrigley_field


The Kaggle dataset provides supplementary ballpark and team level attributes. Similar preprocessing steps are applied, including cleaning park and team names and generating standardized keys (park_grouping and team_key). These transformations ensure compatibility with the Seamheads dataset. Exploratory checks confirm there are no missing values or duplicate records, making this dataset structurally clean. However, inconsistencies in naming conventions between datasets require manual mapping adjustments to align park identifiers prior to merging.

##### dim_ballpark - dimensional ballpark table

In [11]:
# fix park name differences between Kaggle and Seamheads
park_key_fix = {"great_american_ball_park": "great_american_ballpark",
                "guaranteed_rate_field": "guarantee_rate_park",
                "angel_stadium": "angel_stadium_of_anaheim",
                "yankee_stadium": "new_yankee_stadium",
                "oakland_coliseum": "oaklandalameda_county_coliseum",
                "busch_stadium": "busch_stadium_iii"
               }

# standardize Kaggle park keys so they match Seamheads
kaggle_clean["park_grouping"] = kaggle_clean["park_grouping"].replace(park_key_fix)

# remove duplicate Wrigley Field entry (keep Chicago version)
seamheads_unique = ballpark_1nf[~((ballpark_1nf["park_grouping"] == "wrigley_field") &
                                     (ballpark_1nf["city"] != "Chicago"))]

# create the ballpark dimension table
dim_ballpark = kaggle_clean.merge(seamheads_unique[["park_id","park_name","park_name_clean","city","state","latitude","longitude", "park_grouping"]],
                                  on = "park_grouping",
                                  how = "left",
                                  indicator = True
                                 )

# verify join results
print("rows:", len(dim_ballpark))
print(dim_ballpark["_merge"].value_counts())

rows: 30
_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64


The dimensional ballpark table is constructed by merging the cleaned Kaggle and Seamheads datasets using the standardized park_grouping key. Prior to merging, known discrepancies in naming conventions are corrected using a predefined mapping dictionary. A left join is used to retain all Kaggle records while enriching them with Seamheads attributes such as geographic coordinates. Post merge validation confirms that all records successfully matched (both), indicating complete alignment between datasets. This table serves as the central reference for all ballpark level analysis.

#### OpenMeteo

In [12]:
def build_fact_weather_daily(dim_ballpark, start_date = "2023-06-30", end_date = "2023-08-31"):

    # list of weather variables expected in the final dataset
    daily_vars = ["temperature_2m_max",
                  "temperature_2m_min",
                  "precipitation_sum",
                  "windspeed_10m_max",
                  "relative_humidity_2m_mean",
                  ]

    # ensure required columns exist in dim_ballpark before processing
    required = {"park_id", "latitude", "longitude"}
    missing = required - set(dim_ballpark.columns)

    if missing:
        raise ValueError(f"dim_ballpark is missing required columns: {missing}")

    # check for duplicate park_id values (should be 1 row per park)
    dupes = dim_ballpark[dim_ballpark.duplicated("park_id", keep = False)]
    
    if len(dupes) > 0:
        raise ValueError("dim_ballpark has duplicate park_id values.\n"
                         f"Example duplicates:\n{dupes[['park_id','latitude','longitude']].head()}"
                         )
        
    # load raw_weather data generated from data-ingestion 
    weather_raw = pd.read_csv("./data-sources/weather_raw.csv")

    # standardize date column name from time to date and convert date column to datetime
    if "time" in weather_raw.columns:
        weather_raw = weather_raw.rename(columns = {"time": "date"})
    weather_raw["date"] = pd.to_datetime(weather_raw["date"], errors = "coerce") 

    # join weather data with ballpark dimension to attach coordinates
    weather = weather_raw.merge(dim_ballpark[["park_id", "latitude", "longitude"]],
                                on = "park_id",
                                how = "inner"
                                )
    
    # fill any missing data with NAs
    for v in daily_vars:
        if v not in weather.columns:
            weather[v] = pd.NA

    # remove rows with invalid/missing dates and duplicative pard_id and date combos
    weather = weather.dropna(subset = ["date"]).copy() 
    weather = (weather
               .sort_values(["park_id", "date"])
               .drop_duplicates(["park_id", "date"], keep = "last")
               )

    # create surrogate key for each weather record
    weather = weather.reset_index(drop = True)
    weather["weather_id"] = weather.index + 1

    # reorder columns for final fact table structure
    weather = weather[["weather_id", "park_id", "date", "latitude", "longitude"] + daily_vars]

    return weather
    
weather_fact = build_fact_weather_daily(dim_ballpark, "2023-06-30", "2023-08-31")

# verify expected number of days per park
qc_days = weather_fact.groupby("park_id")["date"].nunique().sort_values()
print(qc_days)

park_id
ANA01    63
STP01    63
STL10    63
SFO03    63
SEA03    63
SAN02    63
PIT08    63
PHO01    63
PHI13    63
OAK01    63
NYC21    63
NYC20    63
MIN04    63
MIL06    63
MIA02    63
LOS03    63
KAN06    63
HOU03    63
DET05    63
DEN02    63
CLE08    63
CIN09    63
CHI12    63
CHI11    63
BOS07    63
BAL12    63
ATL03    63
ARL03    63
TOR02    63
WAS11    63
Name: date, dtype: int64


In [13]:
assert weather_fact.duplicated(["park_id", "date"]).sum() == 0 # there should be only one row per date

# cheacking for any missing weather data
missing_parks = set(weather_fact["park_id"]) - set(dim_ballpark["park_id"])
print(missing_parks)
weather_fact[["temperature_2m_max","temperature_2m_min","precipitation_sum","windspeed_10m_max","relative_humidity_2m_mean"]].describe()

set()


,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean
count,1890.000000,1890.000000,1890.000000,1890.000000,1890.000000
mean,84.705714,67.175185,0.121605,11.484709,70.194709
std,9.486297,8.007168,0.309339,3.506105,14.076745
min,61.600000,49.500000,0.000000,4.000000,10.000000
25%,78.300000,61.525000,0.000000,9.100000,65.000000
50%,84.100000,66.100000,0.004000,11.000000,73.000000
75%,90.100000,71.100000,0.098000,13.400000,80.000000
max,116.200000,92.400000,5.425000,36.500000,99.000000


In [14]:
weather_fact.head()

,weather_id,park_id,date,latitude,longitude,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean
0,1,ANA01,2023-06-30,33.80029,-117.882685,80.3,57.0,0.000,9.5,78
1,2,ANA01,2023-07-01,33.80029,-117.882685,81.2,58.9,0.008,10.9,79
2,3,ANA01,2023-07-02,33.80029,-117.882685,81.6,58.7,0.000,11.6,80
3,4,ANA01,2023-07-03,33.80029,-117.882685,79.9,55.8,0.000,10.1,80
4,5,ANA01,2023-07-04,33.80029,-117.882685,78.9,57.0,0.000,9.8,75


Weather data is loaded from a pre-generated raw dataset (weather_raw.csv) created during the data ingestion stage. This dataset contains daily weather observations retrieved from the Open-Meteo API using latitude and longitude coordinates for each ballpark. During transformation, the raw data is filtered to the specified date range and joined with the dimensional ballpark table to associate each record with the correct park identifiers and coordinates. The dataset is then standardized to ensure consistent structure, with one record per park per day. Duplicate records are removed, and a surrogate key (weather_id) is generated to uniquely identify each observation. Data quality checks confirm complete coverage, ensuring each park has the expected number of daily observations and no duplicate park-date combinations.

#### Pybaseball

In [15]:
# load statcast_raw data generated from data-ingestion
raw_statcast = pd.read_csv("./data-sources/statcast_raw.csv")

print("Raw statcast shape:", raw_statcast.shape) # cheack data structure

# clean and standardize statcast dataset
def clean_statcast(df):

    df = df.copy()
    df.columns = df.columns.str.lower().str.strip() # standardize column names
    df["game_date"] = pd.to_datetime(df["game_date"]) # convert dates to date

    # numeric conversions
    numeric_cols = ["release_speed",
                    "release_spin_rate",
                    "launch_speed",
                    "launch_angle",
                    "hit_distance_sc",
                    "pfx_x",
                    "pfx_z",
                    "plate_x",
                    "plate_z"
                    ]

    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors = "coerce")

    # convert identifier columns to integer types for consistency and joins
    df["game_pk"] = pd.to_numeric(df["game_pk"], errors = "coerce").astype("Int64")
    df["batter"] = pd.to_numeric(df["batter"], errors = "coerce").astype("Int64")
    df["pitcher"] = pd.to_numeric(df["pitcher"], errors = "coerce").astype("Int64")

    # categorical columns
    cat_cols = ["pitch_type",
                "events",
                "description",
                "stand",
                "p_throws",
                "home_team",
                "away_team"
                ]

    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # create unique pitch key
    df["pitch_key"] = (df["game_pk"].astype(str)
                       + "-"
                       + df["at_bat_number"].astype(str)
                       + "-"
                       + df["pitch_number"].astype(str)
                       )

    # remove duplicates
    df = df.drop_duplicates("pitch_key")

    return df

statcast_clean = clean_statcast(raw_statcast)
statcast_clean.head()

Raw statcast shape: (233665, 118)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,pitch_key
0,SL,2023-07-06,87.2,-2.32,4.64,"Díaz, Alexis",660688,664747,field_out,hit_into_play,...,3.15,-0.49,0.49,12.3,NaN,NaN,NaN,NaN,NaN,717481-79-3
1,FF,2023-07-06,95.6,-2.39,4.55,"Díaz, Alexis",660688,664747,<NA>,called_strike,...,1.34,0.92,-0.92,9.5,NaN,NaN,NaN,NaN,NaN,717481-79-2
2,FF,2023-07-06,95.2,-2.30,4.54,"Díaz, Alexis",660688,664747,<NA>,ball,...,1.37,0.63,-0.63,11.3,NaN,NaN,NaN,NaN,NaN,717481-79-1
3,SL,2023-07-06,88.5,-2.16,4.76,"Díaz, Alexis",572816,664747,field_out,hit_into_play,...,2.80,-0.36,0.36,12.1,NaN,NaN,NaN,NaN,NaN,717481-78-3
4,FF,2023-07-06,94.0,-2.23,4.60,"Díaz, Alexis",572816,664747,<NA>,foul,...,1.44,0.97,-0.97,8.0,NaN,NaN,NaN,NaN,NaN,717481-78-2


In [16]:
keep_cols = ["pitch_key",
             "game_pk",
             "game_date",
             "home_team",
             "away_team",
             "pitcher",
             "events",
             "description",
             "release_speed",
             "release_spin_rate",
             "launch_speed",
             ]

statcast_clean = statcast_clean[keep_cols]
print(statcast_clean["game_pk"].nunique())
print(statcast_clean["game_date"].min(), statcast_clean["game_date"].max())
statcast_clean.head()

792
2023-06-30 00:00:00 2023-08-31 00:00:00


,pitch_key,game_pk,game_date,home_team,away_team,pitcher,events,description,release_speed,release_spin_rate,launch_speed
0,717481-79-3,717481,2023-07-06,WSH,CIN,664747,field_out,hit_into_play,87.2,2679.0,82.7
1,717481-79-2,717481,2023-07-06,WSH,CIN,664747,<NA>,called_strike,95.6,2560.0,NaN
2,717481-79-1,717481,2023-07-06,WSH,CIN,664747,<NA>,ball,95.2,2509.0,NaN
3,717481-78-3,717481,2023-07-06,WSH,CIN,664747,field_out,hit_into_play,88.5,2743.0,76.7
4,717481-78-2,717481,2023-07-06,WSH,CIN,664747,<NA>,foul,94.0,2589.0,76.4


Statcast data is loaded from a pre-generated raw dataset (statcast_raw.csv) created during the data ingestion stage. This dataset contains pitch level data originally retrieved in chunks to manage performance and query size limitations. During transformation, the data is cleaned by standardizing column names, converting data types, and ensuring consistency across numeric and categorical fields. A unique identifier (pitch_key) is created by combining game level and pitch level attributes, allowing duplicate records to be safely removed. This processed dataset forms the foundation for pitch level and event level analysis.

##### dim_team - dimensional team table

In [17]:
# create team dimension table from Kaggle data
dim_team = (kaggle_clean[["team_name", "team_key"]]
            .drop_duplicates() # ensure unique teams
            .reset_index(drop = True) # add surrogate key for team dimension
           )

dim_team.head()

,team_name,team_key
0,ATL,atl
1,AZ,az
2,BAL,bal
3,BOS,bos
4,CHC,chc


In [18]:
fact_ballpark_attributes = dim_ballpark[["team_key", "park_id", ]].copy() # map teams to ballparks 

# create surrogate key
fact_ballpark_attributes = fact_ballpark_attributes.reset_index(drop = True)
fact_ballpark_attributes["fact_id"] = fact_ballpark_attributes.index + 1

# reorder columns
fact_ballpark_attributes = fact_ballpark_attributes[["fact_id","team_key","park_id"]]

print(fact_ballpark_attributes["park_id"].isin(dim_ballpark["park_id"]).all())
print(fact_ballpark_attributes["team_key"].isin(dim_team["team_key"]).all())

True
True


In [19]:
# build games table
games = (statcast_clean[["game_pk", "game_date", "home_team", "away_team"]]
         .drop_duplicates()
         .copy()
        )

# normalize team abbreviations
team_code_fix = {"AZ": "ARI",
                 "KC": "KCR",
                 "SD": "SDP",
                 "SF": "SFG",
                 "TB": "TBR",
                 "WSH": "WSN",
                 "ATH": "OAK"
                }

games["home_team_std"] = games["home_team"].replace(team_code_fix)
games["away_team_std"] = games["away_team"].replace(team_code_fix)

# canonical team to park map
team_park_map = pd.DataFrame({"home_team_std": ["ARI","ATL","BAL","BOS","CHC","CWS","CIN","CLE","COL","DET",
                                                "HOU","KCR","LAA","LAD","MIA","MIL","MIN","NYY","NYM","OAK",
                                                "PHI","PIT","SDP","SEA","SFG","STL","TBR","TEX","TOR","WSN"
                                               ],
                              "park_id": ["PHO01","ATL03","BAL12","BOS07","CHI11","CHI12","CIN09","CLE08","DEN02","DET05",
                                          "HOU03","KAN06","ANA01","LOS03","MIA02","MIL06","MIN04","NYC21","NYC20","OAK01",
                                          "PHI13","PIT08","SAN02","SEA03","SFO03","STL10","STP01","ARL03","TOR02","WAS11"
                                          ]
                              })

# join
games = games.merge(team_park_map,
                    on = "home_team_std",
                    how = "left",
                    validate = "many_to_one"
                   )
# merge games with weather on park_id and date
games_weather = games.merge(weather_fact,
                            left_on = ["park_id", "game_date"],
                            right_on = ["park_id", "date"],
                            how = "left",
                            validate = "many_to_one"
                           )

games_weather.head()

,game_pk,game_date,home_team,away_team,home_team_std,away_team_std,park_id,weather_id,date,latitude,longitude,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean
0,717481,2023-07-06,WSH,CIN,WSN,CIN,WAS11,1834,2023-07-06,38.872987,-77.007435,86.4,70.9,0.205,5.7,78
1,717480,2023-07-06,CWS,TOR,CWS,TOR,CHI12,385,2023-07-06,41.829892,-87.633703,70.4,64.6,0.039,12.0,89
2,717476,2023-07-06,DET,ATH,DET,OAK,DET05,637,2023-07-06,42.339063,-83.048627,86.1,69.6,0.142,13.7,78
3,717475,2023-07-06,NYY,BAL,NYY,BAL,NYC21,1141,2023-07-06,40.829586,-73.926413,86.9,70.8,0.000,12.4,79
4,717474,2023-07-06,MIL,CHC,MIL,CHC,MIL06,952,2023-07-06,43.028232,-87.970966,73.0,60.4,0.398,8.1,84


In [20]:
# get team of pitcher from statcast data (home vs away can be inferred from inning_topbot and home_team/away_team)
raw_statcast["pitcher_team"] = raw_statcast.apply(lambda row: row["away_team"] if row["inning_topbot"] == "Bot" else row["home_team"], axis = 1)
print(raw_statcast["pitcher_team"])
raw_statcast[["player_name", "pitcher_team", "home_team", "away_team", "inning_topbot"]]

0         CIN
1         CIN
2         CIN
3         CIN
4         CIN
         ... 
233660    MIN
233661    MIN
233662    MIN
233663    MIN
233664    MIN
Name: pitcher_team, Length: 233665, dtype: object


,player_name,pitcher_team,home_team,away_team,inning_topbot
0,"Díaz, Alexis",CIN,WSH,CIN,Bot
1,"Díaz, Alexis",CIN,WSH,CIN,Bot
2,"Díaz, Alexis",CIN,WSH,CIN,Bot
3,"Díaz, Alexis",CIN,WSH,CIN,Bot
4,"Díaz, Alexis",CIN,WSH,CIN,Bot
...,...,...,...,...,...
233660,"Gray, Sonny",MIN,MIN,TEX,Top
233661,"Gray, Sonny",MIN,MIN,TEX,Top
233662,"Gray, Sonny",MIN,MIN,TEX,Top
233663,"Gray, Sonny",MIN,MIN,TEX,Top


#### Final Tables

##### fact_statcast_pitch_table

In [21]:
# create pitch level fact table
fact_statcast_pitch_table = statcast_clean[["pitch_key",
                                            "game_pk",
                                            "pitcher",
                                            "events",
                                            "description",
                                            "release_speed",
                                            "release_spin_rate",
                                            "launch_speed"
                                            ]]

# rename for consistency
fact_statcast_pitch_table = fact_statcast_pitch_table.rename(columns = {"pitcher": "pitcher_id"}).copy()
fact_statcast_pitch_table.head()

,pitch_key,game_pk,pitcher_id,events,description,release_speed,release_spin_rate,launch_speed
0,717481-79-3,717481,664747,field_out,hit_into_play,87.2,2679.0,82.7
1,717481-79-2,717481,664747,<NA>,called_strike,95.6,2560.0,NaN
2,717481-79-1,717481,664747,<NA>,ball,95.2,2509.0,NaN
3,717481-78-3,717481,664747,field_out,hit_into_play,88.5,2743.0,76.7
4,717481-78-2,717481,664747,<NA>,foul,94.0,2589.0,76.4


The fact_statcast_pitch_table is the primary fact table and contains pitch level data for each recorded event. Each row represents a single pitch and is uniquely identified by a composite pitch_key. The table includes performance related attributes such as pitch outcome, velocity, spin rate, and launch metrics. By isolating these high frequency observations in a fact table, the design supports detailed analysis while avoiding redundancy in descriptive attributes.

##### games_table 

In [22]:
# create game level table
games_table = games[['game_pk', 'game_date', 'home_team', 'away_team', 'park_id']]
games_table.head()

,game_pk,game_date,home_team,away_team,park_id
0,717481,2023-07-06,WSH,CIN,WAS11
1,717480,2023-07-06,CWS,TOR,CHI12
2,717476,2023-07-06,DET,ATH,DET05
3,717475,2023-07-06,NYY,BAL,NYC21
4,717474,2023-07-06,MIL,CHC,MIL06


The games_table provides a game level view of the data, with one row per game. It includes the game identifier, date, participating teams, and the ballpark where the game was played. This table acts as a bridge between pitch level data and higher level contextual information, enabling aggregation and analysis at the game level.a

##### dim_pitcher 

In [23]:
unique_ids = fact_statcast_pitch_table['pitcher_id'].unique().tolist() # get unique pitcher IDs

names_df = playerid_reverse_lookup(unique_ids, key_type = 'mlbam') # lookup player names
names_df['player_name'] = names_df['name_first'].str.capitalize() + ' ' + names_df['name_last'].str.capitalize() # create full name

dim_pitcher_table = names_df[['key_mlbam', 'player_name']].copy() # create pitcher dimension table
dim_pitcher_table = dim_pitcher_table.rename(columns = {'key_mlbam': 'pitcher_id'}) # rename key

dim_pitcher= dim_pitcher_table
dim_pitcher.head()

Gathering player lookup table. This may take a moment.


,pitcher_id,player_name
0,642239,Rob Zastryzny
1,670032,Nicky Lopez
2,596001,Jakob Junis
3,593576,Héctor Neris
4,657248,Glenn Otto


The dim_pitcher table stores unique pitcher identifiers along with their corresponding player names. Pitcher IDs are extracted from the pitch level fact table and enriched using an external lookup. By separating pitcher information into a dedicated dimension, the model eliminates repeated player data across pitch records and improves both storage efficiency and query performance.

##### dim_teams 

In [24]:
# create team dimension table with one row per team
dim_teams = kaggle[["team_key", "team_name"]].drop_duplicates()
dim_teams.head()

,team_key,team_name
0,atl,ATL
1,az,AZ
2,bal,BAL
3,bos,BOS
4,chc,CHC


The dim_teams table contains unique team identifiers and team names. It standardizes team representations across datasets and provides a consistent lookup for joining with fact tables. This separation ensures that team related attributes are stored once, rather than repeated across multiple records.

##### dim_ballpark 

In [25]:
dim_ballpark['location_id'] = dim_ballpark['state'] + "_" + dim_ballpark['city'] # create location_id for ballpark table

# create final ballpark dimension table
dim_ballpark_table = dim_ballpark[['park_id', 'park_name', 'longitude', 'latitude', 'location_id']].copy()
display(dim_ballpark_table.head())

,park_id,park_name,longitude,latitude,location_id
0,ATL03,Truist Park,-84.468100,33.891270,GA_Atlanta
1,PHO01,Chase Field,-112.066793,33.445420,AZ_Phoenix
2,BAL12,Oriole Park at Camden Yards,-76.621572,39.283944,MD_Baltimore
3,BOS07,Fenway Park,-71.097337,42.346561,MA_Boston
4,CHI11,Wrigley Field,-87.655397,41.948314,IL_Chicago


The dim_ballpark table stores information about each ballpark, including its identifier, name, and geographic coordinates (lat/long). A derived location_id is used to link ballparks to their corresponding city and state. This table provides location based context for both game level and weather data, supporting spatial analysis.

##### fact_weather 

In [26]:
# create final daily weather fact table
fact_weather = weather_fact[["weather_id",
                             "park_id",
                             'date',
                             'temperature_2m_max',
                             "temperature_2m_min",
                             "precipitation_sum",
                             "windspeed_10m_max",
                             "relative_humidity_2m_mean"]] 
fact_weather.head()

,weather_id,park_id,date,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,relative_humidity_2m_mean
0,1,ANA01,2023-06-30,80.3,57.0,0.000,9.5,78
1,2,ANA01,2023-07-01,81.2,58.9,0.008,10.9,79
2,3,ANA01,2023-07-02,81.6,58.7,0.000,11.6,80
3,4,ANA01,2023-07-03,79.9,55.8,0.000,10.1,80
4,5,ANA01,2023-07-04,78.9,57.0,0.000,9.8,75


The fact_weather table contains daily environmental observations for each ballpark. Each row represents weather conditions for a specific park on a given date, including temperature, precipitation, wind speed, and humidity. This table is structured as a time series fact table and can be joined with game data to analyze the impact of environmental factors on performance.

##### dim_city 

In [27]:
# create city dimension to reduce repeated city/state values
dim_city_table = dim_ballpark[['state', 'city']].copy()
dim_city_table['location_id'] = dim_city_table['state'] + "_" + dim_city_table['city'] # create location key
dim_city_table = dim_city_table.drop_duplicates().reset_index(drop = True) # keep unique locations only
dim_city_table = dim_city_table[['location_id', 'state', 'city']]
dim_city_table.head()

,location_id,state,city
0,GA_Atlanta,GA,Atlanta
1,AZ_Phoenix,AZ,Phoenix
2,MD_Baltimore,MD,Baltimore
3,MA_Boston,MA,Boston
4,IL_Chicago,IL,Chicago


The dim_city table stores unique combinations of city and state, separated from the ballpark table to reduce redundancy. A composite location_id is used as the primary key and links back to the ballpark dimension. This design ensures that location attributes are stored once and reused across the model, aligning with normalization best practices.

#### Saving final tables

In [28]:
# export final tables to CSV files
fact_statcast_pitch_table.to_csv("./cleaned-tables/fact_statcast_pitch.csv", index = False)
games_table.to_csv("./cleaned-tables/games.csv", index = False)
dim_pitcher.to_csv("./cleaned-tables/dim_pitcher.csv", index = False)
dim_teams.to_csv("./cleaned-tables/dim_teams.csv", index = False)
dim_ballpark_table.to_csv("./cleaned-tables/dim_ballpark.csv", index = False)
fact_weather.to_csv("./cleaned-tables/fact_weather.csv", index = False)
dim_city_table.to_csv("./cleaned-tables/dim_city.csv", index = False)